# 08 · Aşama 9 — Ablasyon (profesörün A1–A6 yapılandırmaları)
| | Yapılandırma |
|---|---|
| A1 | Isolation Forest |
| A2 | Autoencoder (ikili, α) |
| A3 | Autoencoder + seçici tespit (üç karar) |
| A4 | Federe AE (ikili) |
| A5 | Federe AE + seçici tespit |
| A6 | INT8 Federe AE + seçici tespit (INT8 yeniden kalibrasyon) |

Tümü tohum 0, aynı test kümeleri. Yaygınlık π = %1 senaryosunda alarm kesinliği de raporlanır.

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
from trustids import conformal as C, evaluation as E
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "final_phase9_ablation")
S0 = E.load_scores(cfg, cfg.train_seeds[0])
FLs = dict(np.load(ART / "fl" / f"fedavg_IID_scores_seed{cfg.train_seeds[0]}.npz"))
Q8 = dict(np.load(ART / "tiny" / "fedavg_int8_scores.npz"))
def evaluate(name, cal, bt, ho, kn, three, model):
    pb, ph, pk = (C.pvalues(cal, x, seed=0, model=model) for x in (bt, ho, kn))
    a, b = cfg.alpha, cfg.alpha + cfg.beta
    fpr = np.mean(pb <= a); ha = np.mean(ph <= a); hu = np.mean((ph > a) & (ph <= b)) if three else 0.0
    bu = np.mean((pb > a) & (pb <= b)) if three else 0.0
    pi = 0.01; prec = (pi * ha) / (pi * ha + (1 - pi) * fpr) if (pi * ha + (1 - pi) * fpr) > 0 else np.nan
    return {"yapılandırma": name, "normal_FPR": fpr, "normal_UNCERTAIN": bu, "görülmemiş_ANOMALY": ha, "görülmemiş_UNCERTAIN": hu,
            "görülmemiş_kaçan": 1 - ha - hu, "bilinen_ANOMALY": np.mean(pk <= a), "alarm_kesinliği_π=%1": prec}
R = [evaluate("A1 · Isolation Forest", S0[("iforest", "benign_cal")], S0[("iforest", "benign_test")], S0[("iforest", "heldout_test")], S0[("iforest", "dev_test")], False, "iforest"),
     evaluate("A2 · AE (ikili)", S0[("ae", "benign_cal")], S0[("ae", "benign_test")], S0[("ae", "heldout_test")], S0[("ae", "dev_test")], False, "ae"),
     evaluate("A3 · AE + seçici", S0[("ae", "benign_cal")], S0[("ae", "benign_test")], S0[("ae", "heldout_test")], S0[("ae", "dev_test")], True, "ae"),
     evaluate("A4 · Federe AE (ikili)", FLs["benign_cal"], FLs["benign_test"], FLs["heldout_test"], FLs["dev_test"], False, "ae"),
     evaluate("A5 · Federe AE + seçici", FLs["benign_cal"], FLs["benign_test"], FLs["heldout_test"], FLs["dev_test"], True, "ae"),
     evaluate("A6 · INT8 Federe AE + seçici", Q8["benign_cal"], Q8["benign_test"], Q8["heldout_test"], Q8["dev_test"], True, "int8")]
AB = pd.DataFrame(R); AB.to_csv(RUN / "08_ablation.csv", index=False)
cfg.save(RUN / "config_used.json"); print("sonuçlar:", RUN); AB.round(4)

**Okuma kılavuzu (profesörün Aşama 9 soruları):** seçici tespit yanlış kararları azaltıyor mu (A2→A3, A4→A5)?
FL görülmemiş saldırıları nasıl etkiliyor (A3↔A5)? INT8 güvenilirliği bozuyor mu (A5↔A6)?
Not: A4–A6'daki federe model, merkezi modelden çok daha kısa eğitildi (aynı tur bütçesi); bu fark yorumlanırken dikkate alınmalıdır.

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")